# 🧪 05 · Ask Your Test Docs (RAG) — Advanced track

**Level:** Experts (but the journey is simple) · **Time:** ~30 min

This teaches **RAG (Retrieval-Augmented Generation)** — the technique behind almost every
serious AI assistant. Instead of the model guessing, it **retrieves the right passages from
your own documents** and answers *from them*, with citations. This is how you stop hallucinations.

The journey is still simple: run the cells, then type a question in the box.

> ⚙️ Enable GPU: *Runtime → Change runtime type → T4 GPU*.

### Step 1 — Install (an embedding model + the chat model)
`sentence-transformers` turns text into vectors so we can find the most relevant passages.

In [ ]:
!pip install -q -U sentence-transformers transformers accelerate
import numpy as np, torch
print('✅ libraries ready')

### Step 2 — Your knowledge base
Below is a small **sample QA knowledge base**. To use *your* team's docs instead, upload
`.txt` files via the 📁 Files panel and load them into `DOCS` (one string per chunk).

In [ ]:
DOCS = [
    "Bug severity scale. S1 Critical: system down, data loss, or security breach with no workaround. S2 Major: key feature broken but a workaround exists. S3 Minor: small issue, low impact. S4 Cosmetic: typo or styling only.",
    "Definition of Done. A story is done when: acceptance criteria are met, code is peer reviewed, unit and integration tests pass, no S1/S2 bugs remain open, and the feature is verified in the staging environment.",
    "Bug report standard. Every bug must include: a clear title, exact steps to reproduce, expected result, actual result, environment (OS/browser/build), severity, and a screenshot or log where relevant.",
    "Regression policy. Before each release we run the full regression suite on staging. Any new S1 or S2 defect blocks the release until fixed and re-verified.",
    "Test data policy. Never use real customer data in test environments. Use synthetic or anonymized data only. Payment testing uses sandbox cards, never live card numbers.",
    "Smoke vs regression. A smoke test is a quick check that core flows work after a build. A regression test is a deeper check that existing features still work after changes.",
]
print(f'✅ {len(DOCS)} documents loaded')

### Step 3 — Index the documents (create embeddings)
Each document becomes a vector. Later we compare your question's vector to these to find matches.

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_emb = embedder.encode(DOCS, normalize_embeddings=True)
print('✅ indexed', len(DOCS), 'documents')

### Step 4 — Load the chat model

In [ ]:
from transformers import pipeline
chat = pipeline('text-generation', model='Qwen/Qwen2.5-1.5B-Instruct',
                torch_dtype=torch.bfloat16, device_map='auto')
print('✅ chat model ready')

### Step 5 — The RAG function (retrieve, then answer with citations)
This is the core idea: find the top matching passages, then ask the model to answer **only**
from those passages and cite them. If the answer isn't in the docs, it should say so.

In [ ]:
SYSTEM = ('You are a precise QA assistant. Answer ONLY using the provided context. '
          'Cite the sources you used like [1], [2]. If the answer is not in the context, '
          'say "I could not find this in the documents." Do not make things up.')

def ask_docs(question, k=3):
    q_emb = embedder.encode([question], normalize_embeddings=True)[0]
    scores = doc_emb @ q_emb                       # cosine similarity (vectors are normalized)
    top = np.argsort(scores)[::-1][:k]             # indices of the k best matches
    context = "\n\n".join(f"[{i+1}] {DOCS[idx]}" for i, idx in enumerate(top))
    user = f"Context:\n{context}\n\nQuestion: {question}"
    out = chat([{'role':'system','content':SYSTEM}, {'role':'user','content':user}],
               max_new_tokens=400, do_sample=False)
    answer = out[0]['generated_text'][-1]['content']
    sources = [DOCS[idx] for idx in top]
    return answer, sources

print('✅ RAG ready')

### ⭐ Easy mode — ask a question in the box
Run this, then type a question like *"What counts as an S1 bug?"* or *"Can we use real customer data in tests?"*

In [ ]:
# 👇 You don't need to read this — just run it and use the box below.
import ipywidgets as widgets
from IPython.display import display, clear_output

q = widgets.Text(value='What counts as an S1 bug?', placeholder='Ask about your QA docs...',
                 layout=widgets.Layout(width='100%'))
go = widgets.Button(description='Ask the docs', button_style='primary')
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        if not q.value.strip():
            print('Type a question first.'); return
        print('⏳ searching...')
        answer, sources = ask_docs(q.value)
        clear_output()
        print('ANSWER:\n' + answer)
        print('\n--- retrieved passages (the evidence) ---')
        for i, s in enumerate(sources, 1):
            print(f'[{i}] {s[:160]}...')

go.on_click(on_click)
display(q, go, out)

---
### 🧠 Expert discussion
- **Why RAG matters:** the model answers from *your* documents, not its training data — so it stays current and you can trace every answer to a source.
- **Chunking:** here each doc is one chunk. For real docs, split long files into ~200–500 word chunks for better retrieval.
- **Try to break it:** ask something *not* in the docs — a good RAG system says "I don't know" instead of inventing an answer. Compare that to asking the plain model in Notebook 01.
- **Scaling up:** swap the numpy similarity for a vector DB (FAISS, Chroma) when you have thousands of chunks.

---
### 🎮 Try this (build intuition)
Questions to copy: [sample_questions.txt](https://github.com/hn-alchemist/qa-ai-workshop/blob/main/data/sample_questions.txt). Challenges: [playground_challenges.md](https://github.com/hn-alchemist/qa-ai-workshop/blob/main/data/playground_challenges.md).
1. **Honesty test:** ask something NOT in the docs (e.g. *"What is the VPN password?"*). Does it say it doesn't know, or invent an answer?
2. Now **delete the "say I don't know" rule** from `SYSTEM` and ask again — see how grounding changes behaviour.
3. **Two-document reasoning:** ask *"If a regression test uses real customer data, which rules did we break?"* — did it cite both sources?

Discuss: what makes RAG more trustworthy than the plain model in Notebook 01?